## RAG Pipelines- Data Ingestion to vector DB Pipeline

In [3]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\mdeht\Desktop\projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## Read all the pdf inside the directory

def process_all_pdfs(pdf_directory):
    """Process all Pdf files in a directory"""
    all_documents =[]
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively
    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} Pdf files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            #ADD source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" X Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("../data")


Found 3 Pdf files to process

Processing: ai and its application.pdf
 Loaded 5 pages

Processing: ai based modelling.pdf
 Loaded 20 pages

Processing: neural network.pdf
 Loaded 83 pages

Total documents loaded: 108


In [5]:
##Text Splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG Performance"""
    text_splitter =RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 108 documents into 364 chunks

Example chunk:
Content: © 2023 IJRTI | Volume 8, Issue 4 | ISSN: 2456-3315 
  
IJRTI2304061 International Journal for Research Trends and Innovation (www.ijrti.org) 356 
 
RESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & 
ITS APP...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-10T10:53:49+05:30', 'author': 'pc 4', 'moddate': '2023-04-10T10:53:49+05:30', 'source': '..\\data\\pdf\\ai and its application.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'ai and its application.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-10T10:53:49+05:30', 'author': 'pc 4', 'moddate': '2023-04-10T10:53:49+05:30', 'source': '..\\data\\pdf\\ai and its application.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'ai and its application.pdf', 'file_type': 'pdf'}, page_content='© 2023 IJRTI | Volume 8, Issue 4 | ISSN: 2456-3315 \n  \nIJRTI2304061 International Journal for Research Trends and Innovation (www.ijrti.org) 356 \n \nRESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & \nITS APPLICATIONS \n \n \nProf. Neha Saini \n \nAssistant Professor in Department of Computer Science & IT \nSDAM College Dinanagar \n \nABSTRACT- \n \nIt is the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to \nthe similar task of using computers to understand human intelligence, but AI does not have to confine itself to methods that \nare biologically obs

Embedding and Vector Store DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
class EmbeddingManager:
    """Handles document embeddings generation using SentenceTransformer"""

    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
        model_name:Hugging face model name for a sentence embeddings

        """
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the sentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model laoded sucessfully. Embedding dimension : {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model{self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) ->np.array:
        """ Generate embeddings for a list of texts

        Args:
             texts: List of text strings embed

        Returns:
        numpy array of embeddings with shape(len(tetxs),embeddings_dim)
          """
        if not self.model:
            raise ValueError("model not loaded")
        
        print(f"Generating embeddings for{len(texts)} texts...")
        embeddings= self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings
    
##Initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 441.41it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model laoded sucessfully. Embedding dimension : 384


VectorStore

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [12]:
## Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

##Generate the embedding

embeddings=embedding_manager.generate_embeddings(texts)

##Store into the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for364 texts...


Batches: 100%|██████████| 12/12 [00:20<00:00,  1.75s/it]


Generated embeddings with shape : (364, 384)
Adding 364 documents to vector store...
Successfully added 364 documents to vector store
Total documents in collection: 728


In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
            
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriver=RAGRetriever(vectorstore,embedding_manager)

In [14]:
rag_retriver

In [15]:
rag_retriver.retrieve("what is neural network is all you need")

Retrieving documents for query: 'what is neural network is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.86it/s]

Generated embeddings with shape : (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_44d2b57f_56',
  'content': 'bilities and the nature of the data, and the desired outcome.\nNeural Networks and\xa0Deep Learning\nDeep learning (DL) [ 80] is known as another popular AI \ntechnique, which is based on artificial neural networks \n(ANN). Nowadays, DL has become a hot topic in the \ncomputing world due to its layer-wise learning capabil-\nity from data. Multiple hidden layers, including input and \noutput layers, make up a typical deep neural network. Fig-\nure\xa0 4 shows a general structure of a deep neural network \n( hidden layer = N and N ≥ 2) comparing with a shallow \nnetwork ( hidden layer = 1 ). DL techniques can be divided \ninto three major categories, highlighted in our earlier paper \nSarker et\xa0al. [80]. These are as below:\n• Deep networks for supervised or discriminative learning \nIn supervised or classification applications, this type of \nDL approach is used to provide a discriminative function. \nDiscriminative deep architectures are ofte

In [16]:
rag_retriver.retrieve("Non-homogeneous Activation Function")

Retrieving documents for query: 'Non-homogeneous Activation Function'
Top K: 5, Score threshold: 0.0
Generating embeddings for1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.35it/s]

Generated embeddings with shape : (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_a92698ab_337',
  'content': 'Gradient Flow Dynamics Beyond the Origin\nF.2 Non-homogeneous Activation Function\nAlthough our theoretical results are stated for homogeneous activations, we evaluate whether\nthe preservation of the sparsity structure also occurs with non-homogeneous activations,\nspeciﬁcally tanh and Gaussian Error Linear Unit (GELU) (Hendrycks and Gimpel, 2016).\nIn all experiments the weights are trained using gradient descent with small initialization,\nand training is continued until the weights escape the origin and reach the next saddle point.\nTanh activation function. We train two-, three-, and four-layer neural network with\ntanh activation function, where the results are depicted in Figure 9, Figure 10 and Fig-\nure 11, respectively. The training set consists of 100 points sampled uniformly from the\nunit sphere in R20, with the corresponding labels generated by a smaller network. In all three\ncases we observe the same qualitative behavior seen fo

Integration VectorDB Context pipeline With LLm output

In [19]:
##Simple Rag pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize the Groq LLM 
groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="groq/compound-mini",temperature=0.1,max_tokens=1024)

##2. Simple RAG function : retrive context + generate response
def rag_simple(query,retriver,llm,top_k=3):
    ##Retrive the context
    results=retriver.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevent context found to answer the question."
    
    ##generate the answer using GROQ LLM

    prompt=f"""Use the following context to answer the question concisely.

        Context:
        {context}

        Question: {query}

        Answer:"""
    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [20]:
answer=rag_simple("Explain the concept of attention mechanism in neural networks.",rag_retriver,llm)
print(answer)

Retrieving documents for query: 'Explain the concept of attention mechanism in neural networks.'
Top K: 3, Score threshold: 0.0
Generating embeddings for1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.34it/s]

Generated embeddings with shape : (1, 384)
Retrieved 2 documents (after filtering)


**Attention mechanism** is a component that lets a neural network dynamically weight different parts of its input when producing an output.  

1. **Key idea** – instead of treating every input element equally, the model computes a relevance score between a **query** (the current processing context) and each **key** (representations of input elements).  
2. **Scoring** – the scores are usually dot‑products (or learned functions) of query and key vectors, optionally scaled to keep gradients stable.  
3. **Normalization** – a soft‑max turns the scores into a probability distribution (attention weights) that sum to 1.  
4. **Weighted sum** – the weights are used to combine the corresponding **value** vectors, producing a **context vector** that emphasizes the most relevant information.  

Mathematically (scaled‑dot‑product attention):

\[
\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V
\]

where \(Q\), \(K\), \(V\) are matrices of queries, keys, and value

Enhanced rag pipeline

In [21]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    } for doc in results]
    
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    
    if return_context:
        output['context'] = context
    
    return output

# Example usage:
result = rag_advanced("What is Case‑Based Reasoning?", rag_retriver, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Case‑Based Reasoning?'
Top K: 3, Score threshold: 0.1
Generating embeddings for1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.69it/s]

Generated embeddings with shape : (1, 384)
Retrieved 3 documents (after filtering)


Answer: Case‑Based Reasoning (CBR) is an AI and cognitive‑science paradigm that solves new problems by retrieving previously stored “cases” – records of earlier problem‑solving experiences – and adapting their solutions to the current situation. It relies on memory‑based inference: the more similar a new problem is to a past case, the more likely its solution will be applicable.
Sources: [{'source': 'ai based modelling.pdf', 'page': 10, 'score': 0.31766951084136963, 'preview': 'case-based reasoners handle new problems by obtaining pre-\nviously stored ’cases’ that describe similar earlier problem-...'}, {'source': 'ai based modelling.pdf', 'page': 10, 'score': 0.31766951084136963, 'preview': 'case-based reasoners handle new problems by obtaining pre-\nviously stored ’cases’ that describe similar earlier problem-...'}, {'source': 'ai based modelling.pdf', 'page': 10, 'score': 0.29648882150650024, 'preview': '[90] explores an expert system modeling for personalized \ndecision-making in m

In [22]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history
    
    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content
        
        confidence = max([doc['similarity_score'] for doc in results]) if results else 0.0
        
        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer
        
        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content
        
        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })
        
        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriver, llm)
result = adv_rag.query("What is Case‑Based Reasoning?", top_k=3, min_score=0.3, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is Case‑Based Reasoning?'
Top K: 3, Score threshold: 0.3
Generating embeddings for1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.78it/s]

Generated embeddings with shape : (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
case-based reasoners handle new problems by obtaining pre-
viously stored ’cases’ that describe similar earlier problem-
solving experiences and customizing their solutions to meet 
new requirements. For example, patient case histories and 
treatments

 are utilized in medical education to assist diag-
nose and treating new patients. Figure  10 shows a general 
architecture of case-based reasoning. CBR research looks at 
the CBR process as a model of human cognition as well as 
a method for developing intelligent systems.
CBR is utilized in a variety of applications. Lamy et al. 
[52], for example, provide a visual case-based reasoning 
strategy for explainable artificial intelligence for breast 
cancer. Gonzalez et al. [30] provide a case-based reason-
ing-based energy optimization technique. Khosravani et al. 
[47] offers a case-based reasoning application in a defect 
detection system for dripper manufacturing. Corrales et al.

case-based reasoners handle new problems by obtaining pre-
viously stored ’cases’ that describe similar earlier problem-
solving experiences and customizing their solutions to meet 
new requirements. For example, patient case histories and 
treatments are utilized in medical education to assist diag-
nose a